<a href="https://colab.research.google.com/github/peterbabulik/TalkToArgon10100102026/blob/main/TalkToArgon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

v1

In [ ]:

import base64
import html
import json
import os
import re
from IPython.display import HTML, FileLink, display
import ipywidgets as widgets
import kaggle_benchmarks as kbench

# ==========================================
# 1. MODEL SETUP & INITIALIZATION
# ==========================================
available_models = list(kbench.llms.keys())
default_model = (
    "google/gemini-4-argon"
    if "google/gemini-4-argon" in available_models
    else available_models[0]
)

chat_history = []

# ==========================================
# 2. UI CONTROLS
# ==========================================
model_dropdown = widgets.Dropdown(
    options=available_models,
    value=default_model,
    description="Model:",
    layout=widgets.Layout(width="420px"),
)

textarea = widgets.Textarea(
    value="",
    placeholder=(
        "Ask a question or paste code here to test with Gemini 4 Argon..."
    ),
    layout=widgets.Layout(width="100%", height="160px"),
)

btn_send = widgets.Button(
    description="Send Prompt / Code",
    button_style="primary",
    icon="paper-plane",
    layout=widgets.Layout(width="190px", height="38px"),
)

btn_save = widgets.Button(
    description="Export to JSON",
    button_style="success",
    icon="save",
    layout=widgets.Layout(width="160px", height="38px"),
)

btn_reset = widgets.Button(
    description="Reset Chat",
    button_style="warning",
    icon="trash",
    layout=widgets.Layout(width="140px", height="38px"),
)

status_label = widgets.HTML(value="")
chat_display = widgets.HTML(
    value=(
        "<div style='color: #888;'>Chat history is empty. Enter your message"
        " above.</div>"
    )
)
download_links_box = widgets.VBox()


# ==========================================
# 3. HELPER FUNCTIONS (DISK EXPORT & DOWNLOADS)
# ==========================================
def save_chat_to_disk():
  if not chat_history:
    return

  last_answer = (
      chat_history[-1]["text"] if chat_history[-1]["role"] == "model" else ""
  )
  export_payload = {
      "model": model_dropdown.value,
      "last_response": last_answer,
      "extracted_code_blocks": re.findall(
          r"```(?:python)?\n(.*?)```", last_answer, re.DOTALL
      ),
      "full_history": chat_history,
  }

  with open(
      "/kaggle/working/argon_output.json", "w", encoding="utf-8"
  ) as outfile:
    json.dump(export_payload, outfile, ensure_ascii=False, indent=2)

  # Extract separate .py files if code is detected
  for idx, block in enumerate(export_payload["extracted_code_blocks"]):
    lines = block.strip().split("\n")
    if lines and lines[0].startswith("%%writefile"):
      filename = lines[0].replace("%%writefile", "").strip()
      content = "\n".join(lines[1:])
      with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    else:
      with open(
          f"/kaggle/working/argon_code_{idx+1}.py", "w", encoding="utf-8"
      ) as f:
        f.write(block)

  # Render download links in UI
  download_widgets = [
      widgets.HTML(
          "<b style='color: #1a73e8;'>📁 Download generated files:</b>"
      )
  ]
  for fname in os.listdir("/kaggle/working"):
    if fname.endswith((".json", ".py")):
      download_widgets.append(
          widgets.HTML(f"<a href='{fname}' target='_blank'>⬇️ {fname}</a>")
      )
  download_links_box.children = download_widgets


# ==========================================
# 4. CHAT RENDERING WITH INLINE COPY BUTTON
# ==========================================
def refresh_chat_view():
  if not chat_history:
    chat_display.value = (
        "<div style='color: #888;'>Chat history is empty.</div>"
    )
    return

  cards = []
  for idx, entry in enumerate(chat_history):
    if entry["role"] == "user":
      safe_user = html.escape(entry["text"]).replace("\n", "<br>")
      cards.append(f"""
            <div style="background-color: #f1f3f4; border-left: 4px solid #1a73e8; padding: 12px 16px; margin: 12px 0; border-radius: 6px; color: #202124;">
                <b style="color: #1a73e8;">🧑 You:</b>
                <div style="margin-top: 6px; white-space: pre-wrap; font-family: monospace; font-size: 13px;">{safe_user}</div>
            </div>
            """)
    else:
      safe_model = html.escape(entry["text"])
      selected_name = entry.get("model_name", "Gemini 4 Argon")

      # Robust clipboard copy command for mobile / sandboxed iframes
      copy_js = (
          f"var t=document.getElementById('raw_out_{idx}');"
          "t.style.display='block'; t.select();"
          "document.execCommand('copy'); t.style.display='none';"
          "this.innerHTML='✅ Copied!';"
          "this.style.background='#2e7d32';"
      )

      cards.append(f"""
            <div style="background-color: #ffffff; border: 1px solid #dadce0; border-left: 4px solid #34a853; padding: 14px 18px; margin: 12px 0; border-radius: 6px; color: #202124;">
                <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 8px;">
                    <b style="color: #137333; font-size: 14px;">🤖 {selected_name}:</b>
                    <button onclick="{copy_js}"
                            style="background-color: #1976d2; color: white; border: none; padding: 6px 12px; border-radius: 4px; cursor: pointer; font-size: 12px; font-weight: bold;">
                        📋 Copy Response
                    </button>
                </div>
                <textarea id="raw_out_{idx}" style="display:none;">{safe_model}</textarea>
                <div style="white-space: pre-wrap; font-family: monospace; font-size: 13px; line-height: 1.5; background: #fafafa; padding: 10px; border-radius: 4px; border: 1px solid #eee;">{safe_model}</div>
            </div>
            """)

  chat_display.value = "".join(cards)


# ==========================================
# 5. BUTTON EVENT HANDLERS
# ==========================================
def on_send_clicked(b):
  user_input = textarea.value.strip()
  if not user_input:
    return

  btn_send.disabled = True
  chosen_model = model_dropdown.value
  status_label.value = (
      f"<span style='color: #e65100; font-weight: bold;'>⏳ Querying"
      f" {chosen_model}...</span>"
  )

  try:
    # Build complete dialog context
    context = (
        "You are an elite coding assistant and software architect. Analyze,"
        " review, and optimize code provided by the user. Maintain full context"
        " across the conversation.\n\n"
    )
    for msg in chat_history:
      role_str = "User" if msg["role"] == "user" else "Assistant"
      context += f"{role_str}:\n{msg['text']}\n\n"
    context += f"User:\n{user_input}\n\nAssistant:"

    # Call LLM via Kaggle proxy (costs $0 to you, deducted from Kaggle AI quota)
    llm = kbench.llms[chosen_model]
    raw_response = llm.prompt(context, temperature=0.2)

    # Save to history
    chat_history.append({"role": "user", "text": user_input})
    chat_history.append({
        "role": "model",
        "text": str(raw_response),
        "model_name": chosen_model,
    })

    save_chat_to_disk()
    textarea.value = ""
    refresh_chat_view()
    status_label.value = (
        "<span style='color: #2e7d32; font-weight: bold;'>✅ Response ready &"
        " saved to /kaggle/working/</span>"
    )
  except Exception as e:
    status_label.value = f"<span style='color: red;'>❌ Error: {str(e)}</span>"
  finally:
    btn_send.disabled = False


def on_save_clicked(b):
  save_chat_to_disk()
  status_label.value = (
      "<span style='color: #1976d2; font-weight: bold;'>💾 Chat history"
      " exported to /kaggle/working/argon_output.json</span>"
  )


def on_reset_clicked(b):
  chat_history.clear()
  refresh_chat_view()
  download_links_box.children = []
  status_label.value = "<span style='color: #666;'>🧹 Chat reset.</span>"


btn_send.on_click(on_send_clicked)
btn_save.on_click(on_save_clicked)
btn_reset.on_click(on_reset_clicked)

# ==========================================
# 6. DISPLAY APP
# ==========================================
action_bar = widgets.HBox(
    [btn_send, btn_save, btn_reset, status_label],
    layout=widgets.Layout(align_items="center", margin="8px 0"),
)
display(
    widgets.VBox([
        model_dropdown,
        textarea,
        action_bar,
        download_links_box,
        chat_display,
    ])
)

v2

In [ ]:
import datetime
import html
import json
import os
from pathlib import Path
import re
from IPython.display import HTML, display
import ipywidgets as widgets
import kaggle_benchmarks as kbench

# Optional: Try importing markdown for rich rendering, fallback to safe pre-wrap
try:
  import markdown

  HAS_MARKDOWN = True
except ImportError:
  HAS_MARKDOWN = False


class KaggleLLMWorkbench:

  def __init__(self, work_dir="/kaggle/working"):
    self.work_dir = Path(work_dir)
    self.work_dir.mkdir(parents=True, exist_ok=True)

    self.chat_history = []
    self.available_models = list(kbench.llms.keys())
    self.default_model = (
        "google/gemini-4-argon"
        if "google/gemini-4-argon" in self.available_models
        else self.available_models[0]
    )

    self._init_ui()
    self._bind_events()

  # ==========================================
  # 1. UI INITIALIZATION
  # ==========================================
  def _init_ui(self):
    # Top Bar Controls
    self.model_dropdown = widgets.Dropdown(
        options=self.available_models,
        value=self.default_model,
        description="<b>Model:</b>",
        layout=widgets.Layout(width="380px"),
    )

    self.temp_slider = widgets.FloatSlider(
        value=0.2,
        min=0.0,
        max=1.5,
        step=0.05,
        description="<b>Temp:</b>",
         readout_format=".2f",
        layout=widgets.Layout(width="280px"),
    )

    self.max_turns_slider = widgets.IntSlider(
        value=10,
        min=1,
        max=50,
        step=1,
        description="<b>Memory (Turns):</b>",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="260px"),
    )

    # Collapsible System Prompt
    self.sys_prompt_input = widgets.Textarea(
        value=(
            "You are an elite coding assistant and software architect. Analyze,"
            " review, and optimize code provided by the user. Maintain full"
            " context across the conversation."
        ),
        layout=widgets.Layout(width="100%", height="70px"),
    )
    self.settings_accordion = widgets.Accordion(
        children=[
            widgets.VBox([
                widgets.HBox([self.temp_slider, self.max_turns_slider]),
                widgets.HTML("<b>System Prompt:</b>"),
                self.sys_prompt_input,
            ])
        ]
    )
    self.settings_accordion.set_title(0, "⚙️ Advanced Configuration & Memory")
    self.settings_accordion.selected_index = None  # Collapsed by default

    # Main Input
    self.textarea = widgets.Textarea(
        value="",
        placeholder=(
            "Ask a question or paste code here... (Tip: Use '%%writefile"
            " filename.py' inside code blocks to auto-name exports)"
        ),
        layout=widgets.Layout(width="100%", height="150px"),
    )

    # Action Buttons
    self.btn_send = widgets.Button(
        description="Send Prompt",
        button_style="primary",
        icon="paper-plane",
        layout=widgets.Layout(width="150px", height="36px"),
    )
    self.btn_undo = widgets.Button(
        description="Undo Last",
        button_style="info",
        icon="undo",
        layout=widgets.Layout(width="120px", height="36px"),
        disabled=True,
    )
    self.btn_save = widgets.Button(
        description="Export JSON/Code",
        button_style="success",
        icon="download",
        layout=widgets.Layout(width="160px", height="36px"),
    )
    self.btn_reset = widgets.Button(
        description="Clear All",
        button_style="danger",
        icon="trash",
        layout=widgets.Layout(width="110px", height="36px"),
    )

    self.status_label = widgets.HTML(
        value="<span style='color: #666;'>Ready.</span>"
    )
    self.download_links_box = widgets.VBox(
        layout=widgets.Layout(margin="8px 0")
    )

    # Scrollable Chat Container
    self.chat_display = widgets.HTML(value=self._get_empty_chat_html())
    self.chat_container = widgets.Box(
        [self.chat_display],
        layout=widgets.Layout(
            max_height="650px",
            overflow_y="auto",
            border="1px solid #e0e0e0",
            padding="10px",
            border_radius="6px",
        ),
    )

  def _bind_events(self):
    self.btn_send.on_click(self.on_send_clicked)
    self.btn_undo.on_click(self.on_undo_clicked)
    self.btn_save.on_click(self.on_save_clicked)
    self.btn_reset.on_click(self.on_reset_clicked)

  # ==========================================
  # 2. CORE LOGIC & PROMPTING
  # ==========================================
  def _build_context(self, new_user_input: str) -> str:
    sys_prompt = self.sys_prompt_input.value.strip()
    context_parts = [f"System:\n{sys_prompt}\n"] if sys_prompt else []

    # Slice history based on max_turns (1 turn = 1 user + 1 model msg)
    max_msgs = self.max_turns_slider.value * 2
    recent_history = (
        self.chat_history[-max_msgs:] if max_msgs > 0 else self.chat_history
    )

    for msg in recent_history:
      role_str = "User" if msg["role"] == "user" else "Assistant"
      context_parts.append(f"{role_str}:\n{msg['text']}\n")

    context_parts.append(f"User:\n{new_user_input}\n\nAssistant:")
    return "\n".join(context_parts)

  def on_send_clicked(self, _):
    user_input = self.textarea.value.strip()
    if not user_input:
      self._set_status("⚠️ Please enter a prompt first.", "#d93025")
      return

    self._toggle_controls(disabled=True)
    chosen_model = self.model_dropdown.value
    self._set_status(
        f"⏳ Querying <b>{chosen_model}</b> (Temp:"
        f" {self.temp_slider.value})...",
        "#e65100",
    )

    try:
      context = self._build_context(user_input)
      llm = kbench.llms[chosen_model]
      raw_response = llm.prompt(context, temperature=self.temp_slider.value)
      response_text = str(raw_response)

      timestamp = datetime.datetime.now().strftime("%H:%M:%S")
      self.chat_history.append(
          {"role": "user", "text": user_input, "time": timestamp}
      )
      self.chat_history.append({
          "role": "model",
          "text": response_text,
          "model_name": chosen_model,
          "time": timestamp,
      })

      self.textarea.value = ""
      self.refresh_chat_view()
      self.save_chat_to_disk()
      self._set_status(
          f"✅ Response received ({timestamp}) & auto-saved.", "#2e7d32"
      )
    except Exception as e:
      self._set_status(f"❌ Error: { html.escape(str(e)) }", "#c62828")
    finally:
      self._toggle_controls(disabled=False)

  def on_undo_clicked(self, _):
    if len(self.chat_history) >= 2:
      # Pop model response and user prompt, restore user prompt to textarea
      self.chat_history.pop()
      last_user_msg = self.chat_history.pop()
      self.textarea.value = last_user_msg["text"]
      self.refresh_chat_view()
      self._set_status(
          "↩️ Last turn undone. Prompt restored to input box.", "#1976d2"
      )
    self.btn_undo.disabled = len(self.chat_history) < 2

  def on_save_clicked(self, _):
    if not self.chat_history:
      self._set_status("⚠️ Nothing to export yet.", "#e65100")
      return
    self.save_chat_to_disk()
    self._set_status(f"💾 Exported to {self.work_dir}", "#1976d2")

  def on_reset_clicked(self, _):
    self.chat_history.clear()
    self.refresh_chat_view()
    self.download_links_box.children = []
    self.btn_undo.disabled = True
    self._set_status("🧹 Chat history cleared.", "#666")

  # ==========================================
  # 3. DISK EXPORT & CODE EXTRACTION
  # ==========================================
  def save_chat_to_disk(self):
    if not self.chat_history:
      return

    last_answer = (
        self.chat_history[-1]["text"]
        if self.chat_history[-1]["role"] == "model"
        else ""
    )

    # Universal regex matching ```lang \n code ```
    code_blocks = re.findall(
        r"```([a-zA-Z0-9_+-]*)\n(.*?)```", last_answer, re.DOTALL
    )

    export_payload = {
        "timestamp": datetime.datetime.now().isoformat(),
        "model": self.model_dropdown.value,
        "system_prompt": self.sys_prompt_input.value,
        "temperature": self.temp_slider.value,
        "last_response": last_answer,
        "extracted_code_blocks": [
            {"lang": lang or "txt", "code": code} for lang, code in code_blocks
        ],
        "full_history": self.chat_history,
    }

    json_path = self.work_dir / "argon_output.json"
    with open(json_path, "w", encoding="utf-8") as outfile:
      json.dump(export_payload, outfile, ensure_ascii=False, indent=2)

    ext_map = {
        "python": "py",
        "py": "py",
        "javascript": "js",
        "js": "js",
        "html": "html",
        "css": "css",
        "bash": "sh",
        "sh": "sh",
        "sql": "sql",
        "json": "json",
        "cpp": "cpp",
    }
    saved_files = ["argon_output.json"]

    for idx, (lang, block) in enumerate(code_blocks):
      lines = block.strip().split("\n")
      ext = ext_map.get(lang.lower().strip(), "txt")

      # Securely handle %%writefile directive
      if lines and lines[0].strip().startswith("%%writefile"):
        raw_fname = lines[0].replace("%%writefile", "").strip()
        safe_fname = os.path.basename(raw_fname)  # Prevent path traversal
        if not safe_fname:
          safe_fname = f"extracted_{idx+1}.{ext}"
        content = "\n".join(lines[1:])
      else:
        safe_fname = f"argon_code_{idx+1}.{ext}"
        content = block

      file_path = self.work_dir / safe_fname
      with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)
      saved_files.append(safe_fname)

    # Render download links
    links_html = " &nbsp;|&nbsp; ".join([
        f"<a href='{fname}' target='_blank' style='text-decoration:none;"
        f" font-weight:500;'>📄 {fname}</a>"
        for fname in saved_files
    ])
    self.download_links_box.children = [
        widgets.HTML(
            "<div style='background:#e8f0fe; padding:8px 12px;"
            " border-radius:4px; font-size:13px;'><b"
            " style='color:#1967d2;'>📁 Generated Files:</b> "
            + links_html
            + "</div>"
        )
    ]

  # ==========================================
  # 4. VIEW RENDERING & HELPERS
  # ==========================================
  def _get_copy_js(self, element_id: str, btn_text: str = "Copied!") -> str:
    return (
        f"var t=document.getElementById('{element_id}');"
        "t.style.display='block'; t.select(); document.execCommand('copy');"
        " t.style.display='none'; var orig=this.innerHTML;"
        f" this.innerHTML='✅ {btn_text}'; this.style.background='#2e7d32';"
        " var btn=this; setTimeout(function(){ btn.innerHTML=orig;"
        " btn.style.background='#1976d2'; }, 2000);"
    )

  def _format_model_text(self, text: str, msg_idx: int) -> str:
    """Renders markdown or safely formats text with isolated copyable code blocks."""
    if HAS_MARKDOWN:
      return markdown.markdown(text, extensions=["fenced_code", "tables"])
    else:
      return (
          f"<div style='white-space: pre-wrap; font-family: monospace;'>"
          f"{html.escape(text)}</div>"
      )

  def refresh_chat_view(self):
    if not self.chat_history:
      self.chat_display.value = self._get_empty_chat_html()
      return

    cards = ["""
        <style>
            .chat-card pre { background: #282c34; color: #abb2bf; padding: 12px; border-radius: 6px; overflow-x: auto; font-family: monospace; font-size: 13px; line-height: 1.45; }
            .chat-card code { font-family: monospace; background: rgba(0,0,0,0.06); padding: 2px 4px; border-radius: 3px; }
            .chat-card pre code { background: none; padding: 0; }
            .chat-card table { border-collapse: collapse; margin: 10px 0; }
            .chat-card th, .chat-card td { border: 1px solid #ddd; padding: 6px 10px; }
        </style>
        """]

    for idx, entry in enumerate(self.chat_history):
      time_str = entry.get("time", "")
      if entry["role"] == "user":
        safe_user = html.escape(entry["text"])
        cards.append(f"""
                <div class="chat-card" style="background-color: #f8f9fa; border-left: 4px solid #1a73e8; padding: 10px 14px; margin: 10px 0; border-radius: 6px; color: #202124;">
                    <div style="display:flex; justify-content:space-between; color: #1a73e8; font-size: 13px; margin-bottom: 4px;">
                        <b>🧑 You</b> <span style="color:#888; font-size:11px;">{time_str}</span>
                    </div>
                    <div style="white-space: pre-wrap; font-family: monospace; font-size: 13px;">{safe_user}</div>
                </div>
                """)
      else:
        raw_escaped = html.escape(entry["text"])
        rendered_body = self._format_model_text(entry["text"], idx)
        selected_name = entry.get("model_name", "Model")
        copy_js = self._get_copy_js(f"raw_out_{idx}", "Copied Full!")

        cards.append(f"""
                <div class="chat-card" style="background-color: #ffffff; border: 1px solid #dadce0; border-left: 4px solid #34a853; padding: 12px 16px; margin: 10px 0; border-radius: 6px; color: #202124;">
                    <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 8px; border-bottom: 1px solid #f1f3f4; padding-bottom: 6px;">
                        <div>
                            <b style="color: #137333; font-size: 14px;">🤖 {selected_name}</b>
                            <span style="color:#888; font-size:11px; margin-left:8px;">{time_str}</span>
                        </div>
                        <button onclick="{copy_js}" style="background-color: #1976d2; color: white; border: none; padding: 5px 10px; border-radius: 4px; cursor: pointer; font-size: 11px; font-weight: bold;">
                            📋 Copy Raw Markdown
                        </button>
                    </div>
                    <textarea id="raw_out_{idx}" style="display:none;">{raw_escaped}</textarea>
                    <div style="font-size: 13.5px; line-height: 1.6;">{rendered_body}</div>
                </div>
                """)

    self.chat_display.value = "".join(cards)

  def _get_empty_chat_html(self) -> str:
    return (
        "<div style='color: #888; text-align: center; padding: 30px"
        " 0;'>💬 Chat history is empty. Enter your prompt above to begin.</div>"
    )

  def _set_status(self, html_msg: str, color: str = "#202124"):
    self.status_label.value = (
        f"<span style='color: {color}; font-size: 13px;'>{html_msg}</span>"
    )

  def _toggle_controls(self, disabled: bool):
    self.btn_send.disabled = disabled
    self.btn_save.disabled = disabled
    self.btn_reset.disabled = disabled
    self.btn_undo.disabled = disabled or (len(self.chat_history) < 2)

  def display(self):
    top_bar = widgets.HBox(
        [self.model_dropdown], layout=widgets.Layout(margin="0 0 6px 0")
    )
    action_bar = widgets.HBox(
        [
            self.btn_send,
            self.btn_undo,
            self.btn_save,
            self.btn_reset,
            self.status_label,
        ],
        layout=widgets.Layout(align_items="center", margin="8px 0", gap="8px"),
    )
    app_layout = widgets.VBox([
        top_bar,
        self.settings_accordion,
        self.textarea,
        action_bar,
        self.download_links_box,
        self.chat_container,
    ])
    display(app_layout)


# ==========================================
# 5. LAUNCH WORKBENCH
# ==========================================
workbench = KaggleLLMWorkbench()
workbench.display()

V3

In [ ]:

import datetime
import html
import json
import os
from pathlib import Path
import re
import sys
import traceback
from IPython import get_ipython
from IPython.display import HTML, display
from IPython.utils.capture import capture_output
import ipywidgets as widgets
import kaggle_benchmarks as kbench

# Optional: Try importing markdown for rich rendering, fallback to safe pre-wrap
try:
  import markdown

  HAS_MARKDOWN = True
except ImportError:
  HAS_MARKDOWN = False


class KaggleLLMWorkbench:

  def __init__(self, work_dir="/kaggle/working"):
    self.work_dir = Path(work_dir)
    self.work_dir.mkdir(parents=True, exist_ok=True)

    self.chat_history = []
    self._last_exec_log = ""
    self.available_models = list(kbench.llms.keys())
    self.default_model = (
        "google/gemini-4-argon"
        if "google/gemini-4-argon" in self.available_models
        else self.available_models[0]
    )

    self._init_ui()
    self._bind_events()

  # ==========================================
  # 1. UI INITIALIZATION
  # ==========================================
  def _init_ui(self):
    # Top Bar Controls
    self.model_dropdown = widgets.Dropdown(
        options=self.available_models,
        value=self.default_model,
        description="<b>Model:</b>",
        layout=widgets.Layout(width="380px"),
    )

    self.temp_slider = widgets.FloatSlider(
        value=0.2,
        min=0.0,
        max=1.5,
        step=0.05,
        description="<b>Temp:</b>",
        readout_format=".2f",
        layout=widgets.Layout(width="280px"),
    )

    self.max_turns_slider = widgets.IntSlider(
        value=10,
        min=1,
        max=50,
        step=1,
        description="<b>Memory (Turns):</b>",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="260px"),
    )

    # Collapsible System Prompt
    self.sys_prompt_input = widgets.Textarea(
        value=(
            "You are an elite coding assistant and software architect. Analyze,"
            " review, and optimize code provided by the user. Maintain full"
            " context across the conversation."
        ),
        layout=widgets.Layout(width="100%", height="70px"),
    )
    self.settings_accordion = widgets.Accordion(
        children=[
            widgets.VBox([
                widgets.HBox([self.temp_slider, self.max_turns_slider]),
                widgets.HTML("<b>System Prompt:</b>"),
                self.sys_prompt_input,
            ])
        ]
    )
    self.settings_accordion.set_title(0, "⚙️ Advanced Configuration & Memory")
    self.settings_accordion.selected_index = None  # Collapsed by default

    # Main Input
    self.textarea = widgets.Textarea(
        value="",
        placeholder=(
            "Ask a question or paste code here... (Tip: Use '%%writefile"
            " filename.py' inside code blocks to auto-name exports)"
        ),
        layout=widgets.Layout(width="100%", height="150px"),
    )

    # Action Buttons
    self.btn_send = widgets.Button(
        description="Send Prompt",
        button_style="primary",
        icon="paper-plane",
        layout=widgets.Layout(width="150px", height="36px"),
    )
    self.btn_undo = widgets.Button(
        description="Undo Last",
        button_style="info",
        icon="undo",
        layout=widgets.Layout(width="120px", height="36px"),
        disabled=True,
    )
    self.btn_save = widgets.Button(
        description="Export JSON/Code",
        button_style="success",
        icon="download",
        layout=widgets.Layout(width="160px", height="36px"),
    )
    self.btn_reset = widgets.Button(
        description="Clear All",
        button_style="danger",
        icon="trash",
        layout=widgets.Layout(width="110px", height="36px"),
    )

    self.status_label = widgets.HTML(
        value="<span style='color: #666;'>Ready.</span>"
    )
    self.download_links_box = widgets.VBox(
        layout=widgets.Layout(margin="4px 0")
    )

    # --- NEW: Execution Engine UI Controls ---
    self.exec_buttons_box = widgets.HBox(
        layout=widgets.Layout(margin="4px 0", gap="8px", flex_wrap="wrap")
    )

    self.exec_output = widgets.Output(
        layout=widgets.Layout(
            max_height="350px",
            overflow_y="auto",
            border="1px solid #d0d7de",
            padding="10px",
            background_color="#f6f8fa",
            border_radius="6px",
        )
    )

    self.btn_feed_output = widgets.Button(
        description="Send Output/Error to AI",
        icon="reply",
        button_style="warning",
        layout=widgets.Layout(width="210px", height="30px"),
    )
    self.btn_clear_console = widgets.Button(
        description="Hide Console",
        icon="eye-slash",
        layout=widgets.Layout(width="130px", height="30px"),
    )

    self.exec_container = widgets.VBox(
        [
            widgets.HTML("<b>⚡ Execution Console:</b>"),
            self.exec_output,
            widgets.HBox(
                [self.btn_feed_output, self.btn_clear_console],
                layout=widgets.Layout(margin="6px 0 0 0", gap="8px"),
            ),
        ],
        layout=widgets.Layout(display="none", margin="8px 0"),
    )

    # Scrollable Chat Container
    self.chat_display = widgets.HTML(value=self._get_empty_chat_html())
    self.chat_container = widgets.Box(
        [self.chat_display],
        layout=widgets.Layout(
            max_height="650px",
            overflow_y="auto",
            border="1px solid #e0e0e0",
            padding="10px",
            border_radius="6px",
        ),
    )

  def _bind_events(self):
    self.btn_send.on_click(self.on_send_clicked)
    self.btn_undo.on_click(self.on_undo_clicked)
    self.btn_save.on_click(self.on_save_clicked)
    self.btn_reset.on_click(self.on_reset_clicked)
    self.btn_feed_output.on_click(self._on_feed_output_clicked)
    self.btn_clear_console.on_click(
        lambda _: setattr(self.exec_container.layout, "display", "none")
    )

  # ==========================================
  # 2. CORE LOGIC & PROMPTING
  # ==========================================
  def _build_context(self, new_user_input: str) -> str:
    sys_prompt = self.sys_prompt_input.value.strip()
    context_parts = [f"System:\n{sys_prompt}\n"] if sys_prompt else []

    max_msgs = self.max_turns_slider.value * 2
    recent_history = (
        self.chat_history[-max_msgs:] if max_msgs > 0 else self.chat_history
    )

    for msg in recent_history:
      role_str = "User" if msg["role"] == "user" else "Assistant"
      context_parts.append(f"{role_str}:\n{msg['text']}\n")

    context_parts.append(f"User:\n{new_user_input}\n\nAssistant:")
    return "\n".join(context_parts)

  def on_send_clicked(self, _):
    user_input = self.textarea.value.strip()
    if not user_input:
      self._set_status("⚠️ Please enter a prompt first.", "#d93025")
      return

    self._toggle_controls(disabled=True)
    chosen_model = self.model_dropdown.value
    self._set_status(
        f"⏳ Querying <b>{chosen_model}</b> (Temp:"
        f" {self.temp_slider.value})...",
        "#e65100",
    )

    try:
      context = self._build_context(user_input)
      llm = kbench.llms[chosen_model]
      raw_response = llm.prompt(context, temperature=self.temp_slider.value)
      response_text = str(raw_response)

      timestamp = datetime.datetime.now().strftime("%H:%M:%S")
      self.chat_history.append(
          {"role": "user", "text": user_input, "time": timestamp}
      )
      self.chat_history.append({
          "role": "model",
          "text": response_text,
          "model_name": chosen_model,
          "time": timestamp,
      })

      self.textarea.value = ""
      self.refresh_chat_view()
      self.save_chat_to_disk()
      self._set_status(
          f"✅ Response received ({timestamp}) & auto-saved.", "#2e7d32"
      )
    except Exception as e:
      self._set_status(f"❌ Error: {html.escape(str(e))}", "#c62828")
    finally:
      self._toggle_controls(disabled=False)

  def on_undo_clicked(self, _):
    if len(self.chat_history) >= 2:
      self.chat_history.pop()
      last_user_msg = self.chat_history.pop()
      self.textarea.value = last_user_msg["text"]
      self.refresh_chat_view()
      self._set_status(
          "↩️ Last turn undone. Prompt restored to input box.", "#1976d2"
      )
    self.btn_undo.disabled = len(self.chat_history) < 2

  def on_save_clicked(self, _):
    if not self.chat_history:
      self._set_status("⚠️ Nothing to export yet.", "#e65100")
      return
    self.save_chat_to_disk()
    self._set_status(f"💾 Exported to {self.work_dir}", "#1976d2")

  def on_reset_clicked(self, _):
    self.chat_history.clear()
    self.refresh_chat_view()
    self.download_links_box.children = []
    self.exec_buttons_box.children = []
    self.exec_container.layout.display = "none"
    self.btn_undo.disabled = True
    self._set_status("🧹 Chat history cleared.", "#666")

  # ==========================================
  # 3. DISK EXPORT, EXTRACTION & CODE EXECUTION
  # ==========================================
  def save_chat_to_disk(self):
    if not self.chat_history:
      return

    last_answer = (
        self.chat_history[-1]["text"]
        if self.chat_history[-1]["role"] == "model"
        else ""
    )
    code_blocks = re.findall(
        r"```([a-zA-Z0-9_+-]*)\n(.*?)```", last_answer, re.DOTALL
    )

    export_payload = {
        "timestamp": datetime.datetime.now().isoformat(),
        "model": self.model_dropdown.value,
        "system_prompt": self.sys_prompt_input.value,
        "temperature": self.temp_slider.value,
        "last_response": last_answer,
        "extracted_code_blocks": [
            {"lang": lang or "txt", "code": code} for lang, code in code_blocks
        ],
        "full_history": self.chat_history,
    }

    json_path = self.work_dir / "argon_output.json"
    with open(json_path, "w", encoding="utf-8") as outfile:
      json.dump(export_payload, outfile, ensure_ascii=False, indent=2)

    ext_map = {
        "python": "py",
        "py": "py",
        "javascript": "js",
        "js": "js",
        "html": "html",
        "css": "css",
        "bash": "sh",
        "sh": "sh",
        "sql": "sql",
        "json": "json",
        "cpp": "cpp",
    }
    saved_files = ["argon_output.json"]

    for idx, (lang, block) in enumerate(code_blocks):
      lines = block.strip().split("\n")
      ext = ext_map.get(lang.lower().strip(), "txt")

      if lines and lines[0].strip().startswith("%%writefile"):
        raw_fname = lines[0].replace("%%writefile", "").strip()
        safe_fname = os.path.basename(raw_fname)
        if not safe_fname:
          safe_fname = f"extracted_{idx+1}.{ext}"
        content = "\n".join(lines[1:])
      else:
        safe_fname = f"argon_code_{idx+1}.{ext}"
        content = block

      file_path = self.work_dir / safe_fname
      with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)
      saved_files.append(safe_fname)

    # Render download links
    links_html = " &nbsp;|&nbsp; ".join([
        f"<a href='{fname}' target='_blank' style='text-decoration:none;"
        f" font-weight:500;'>📄 {fname}</a>"
        for fname in saved_files
    ])
    self.download_links_box.children = [
        widgets.HTML(
            "<div style='background:#e8f0fe; padding:8px 12px;"
            " border-radius:4px; font-size:13px;'><b"
            " style='color:#1967d2;'>📁 Generated Files:</b> "
            + links_html
            + "</div>"
        )
    ]

    # Dynamically generate "Run" buttons for extracted Python and Bash scripts
    run_buttons = []
    for fname in saved_files:
      if fname.endswith((".py", ".sh")):
        btn = widgets.Button(
            description=f"▶️ Run {fname}",
            button_style="success",
            tooltip=f"Execute {fname} in current notebook session",
            layout=widgets.Layout(width="auto", height="30px"),
        )
        btn.on_click(lambda _, f=fname: self.execute_file(f))
        run_buttons.append(btn)

    self.exec_buttons_box.children = run_buttons

  def execute_file(self, filename: str):
    file_path = self.work_dir / filename
    if not file_path.exists():
      self._set_status(f"⚠️ File {filename} not found.", "#d93025")
      return

    code = file_path.read_text(encoding="utf-8")
    self.exec_container.layout.display = "block"
    self.exec_output.clear_output(wait=True)
    self._set_status(f"⚡ Executing <b>{filename}</b>...", "#e65100")

    ip = get_ipython()
    exec_success = True

    with self.exec_output:
      print(f"--- [Executing {filename}] ---")
      with capture_output(stdout=True, stderr=True, display=True) as captured:
        try:
          if filename.endswith(".sh"):
            ip.system(f"bash {file_path}")
          else:
            res = ip.run_cell(code)
            if res.error_in_exec:
              exec_success = False
            elif res.error_before_exec:
              exec_success = False
        except Exception:
          traceback.print_exc()
          exec_success = False

      # Re-emit captured output to the widget so prints, tables, and plots render
      captured.show()

    # Format logs for the "Send Output/Error to AI" button
    log_parts = []
    if captured.stdout:
      log_parts.append(f"STDOUT:\n{captured.stdout.strip()}")
    if captured.stderr:
      log_parts.append(f"STDERR / TRACEBACK:\n{captured.stderr.strip()}")
    log_str = (
        "\n\n".join(log_parts)
        if log_parts
        else "Code executed silently with no stdout/stderr output."
    )

    self._last_exec_log = (
        f"I executed `{filename}` and got the following"
        f" output:\n```text\n{log_str[-4000:]}\n```"
    )

    if exec_success:
      self._set_status(
          f"✅ Execution of <b>{filename}</b> finished.", "#2e7d32"
      )
    else:
      self._set_status(
          f"❌ Execution failed in <b>{filename}</b>. Click 'Send Output/Error"
          " to AI' to debug.",
          "#c62828",
      )

  def _on_feed_output_clicked(self, _):
    if self._last_exec_log:
      current_text = self.textarea.value.strip()
      prefix = f"{current_text}\n\n" if current_text else ""
      self.textarea.value = (
          f"{prefix}{self._last_exec_log}\n\nPlease analyze the result or fix"
          " any errors:"
      )
      self._set_status("📋 Execution log appended to prompt box.", "#1976d2")

  # ==========================================
  # 4. VIEW RENDERING & HELPERS
  # ==========================================
  def _get_copy_js(self, element_id: str, btn_text: str = "Copied!") -> str:
    return (
        f"var t=document.getElementById('{element_id}');"
        "t.style.display='block'; t.select(); document.execCommand('copy');"
        " t.style.display='none'; var orig=this.innerHTML;"
        f" this.innerHTML='✅ {btn_text}'; this.style.background='#2e7d32';"
        " var btn=this; setTimeout(function(){ btn.innerHTML=orig;"
        " btn.style.background='#1976d2'; }, 2000);"
    )

  def _format_model_text(self, text: str, msg_idx: int) -> str:
    if HAS_MARKDOWN:
      return markdown.markdown(text, extensions=["fenced_code", "tables"])
    else:
      return (
          f"<div style='white-space: pre-wrap; font-family: monospace;'>"
          f"{html.escape(text)}</div>"
      )

  def refresh_chat_view(self):
    if not self.chat_history:
      self.chat_display.value = self._get_empty_chat_html()
      return

    cards = ["""
        <style>
            .chat-card pre { background: #282c34; color: #abb2bf; padding: 12px; border-radius: 6px; overflow-x: auto; font-family: monospace; font-size: 13px; line-height: 1.45; }
            .chat-card code { font-family: monospace; background: rgba(0,0,0,0.06); padding: 2px 4px; border-radius: 3px; }
            .chat-card pre code { background: none; padding: 0; }
            .chat-card table { border-collapse: collapse; margin: 10px 0; }
            .chat-card th, .chat-card td { border: 1px solid #ddd; padding: 6px 10px; }
        </style>
        """]

    for idx, entry in enumerate(self.chat_history):
      time_str = entry.get("time", "")
      if entry["role"] == "user":
        safe_user = html.escape(entry["text"])
        cards.append(f"""
                <div class="chat-card" style="background-color: #f8f9fa; border-left: 4px solid #1a73e8; padding: 10px 14px; margin: 10px 0; border-radius: 6px; color: #202124;">
                    <div style="display:flex; justify-content:space-between; color: #1a73e8; font-size: 13px; margin-bottom: 4px;">
                        <b>🧑 You</b> <span style="color:#888; font-size:11px;">{time_str}</span>
                    </div>
                    <div style="white-space: pre-wrap; font-family: monospace; font-size: 13px;">{safe_user}</div>
                </div>
                """)
      else:
        raw_escaped = html.escape(entry["text"])
        rendered_body = self._format_model_text(entry["text"], idx)
        selected_name = entry.get("model_name", "Model")
        copy_js = self._get_copy_js(f"raw_out_{idx}", "Copied Full!")

        cards.append(f"""
                <div class="chat-card" style="background-color: #ffffff; border: 1px solid #dadce0; border-left: 4px solid #34a853; padding: 12px 16px; margin: 10px 0; border-radius: 6px; color: #202124;">
                    <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 8px; border-bottom: 1px solid #f1f3f4; padding-bottom: 6px;">
                        <div>
                            <b style="color: #137333; font-size: 14px;">🤖 {selected_name}</b>
                            <span style="color:#888; font-size:11px; margin-left:8px;">{time_str}</span>
                        </div>
                        <button onclick="{copy_js}" style="background-color: #1976d2; color: white; border: none; padding: 5px 10px; border-radius: 4px; cursor: pointer; font-size: 11px; font-weight: bold;">
                            📋 Copy Raw Markdown
                        </button>
                    </div>
                    <textarea id="raw_out_{idx}" style="display:none;">{raw_escaped}</textarea>
                    <div style="font-size: 13.5px; line-height: 1.6;">{rendered_body}</div>
                </div>
                """)

    self.chat_display.value = "".join(cards)

  def _get_empty_chat_html(self) -> str:
    return (
        "<div style='color: #888; text-align: center; padding: 30px"
        " 0;'>💬 Chat history is empty. Enter your prompt above to begin.</div>"
    )

  def _set_status(self, html_msg: str, color: str = "#202124"):
    self.status_label.value = (
        f"<span style='color: {color}; font-size: 13px;'>{html_msg}</span>"
    )

  def _toggle_controls(self, disabled: bool):
    self.btn_send.disabled = disabled
    self.btn_save.disabled = disabled
    self.btn_reset.disabled = disabled
    self.btn_undo.disabled = disabled or (len(self.chat_history) < 2)

  def display(self):
    top_bar = widgets.HBox(
        [self.model_dropdown], layout=widgets.Layout(margin="0 0 6px 0")
    )
    action_bar = widgets.HBox(
        [
            self.btn_send,
            self.btn_undo,
            self.btn_save,
            self.btn_reset,
            self.status_label,
        ],
        layout=widgets.Layout(align_items="center", margin="8px 0", gap="8px"),
    )
    app_layout = widgets.VBox([
        top_bar,
        self.settings_accordion,
        self.textarea,
        action_bar,
        self.download_links_box,
        self.exec_buttons_box,
        self.exec_container,
        self.chat_container,
    ])
    display(app_layout)


# ==========================================
# 5. LAUNCH WORKBENCH
# ==========================================
workbench = KaggleLLMWorkbench()
workbench.display()